In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np 
from camoufox.async_api import AsyncCamoufox
import re
from bs4 import BeautifulSoup
import requests

In [ ]:
base_url = "https://divar.ir/s/iran/car?q=%D9%85%D8%A7%D8%B4%DB%8C%D9%86"
car_url = []

async with AsyncCamoufox() as browser :
    page = await browser.new_page()
    await page.goto(base_url)
    for i in range(100):
        await page.evaluate('window.scrollBy(0, document.body.scrollHeight)')
        await page.wait_for_timeout(1000)
        urls = links = await page.locator("a.kt-post-card__action").all()
        await page.wait_for_timeout(1000)
        for u in urls:
            text = await u.get_attribute("href")
            car_url.append(text)
        await page.wait_for_timeout(500)  
        button = page.locator('button.post-list__load-more-btn-be092')
        if await button.count() > 0 :
            await button.click()
            await page.wait_for_timeout(600)
        

In [ ]:
len(car_url)

In [ ]:
car_url = np.unique(car_url)

In [ ]:
len(car_url)

In [ ]:
title = []
brand = []
gerbox = []
fuel = []
price = []
bemaeh = []
km = []
colors = []
motor_status = []
Chassis_status = []
body_status = []
gerbox_status = []
color_re = re.compile(
    r"(سفید|مشکی|سیاه|نقره[\s‌]?ای|خاکستری|طوسی|قرمز|سرخ|آبی|سبز|زرد|"
    r"نارنجی|قهوه[\s‌]?ای|بژ|شرابی|طلایی|سرمه[\s‌]?ای|کرم|صورتی|بنفش|"
    r"نوک[\s‌]?مدادی|دو[\s‌]?رنگ)"
)
async with AsyncCamoufox() as browser :
    page = await browser.new_page()
    for url in car_url : 
        await page.goto(f"https://divar.ir{url}")
        await page.wait_for_timeout(2000)
        t = await page.query_selector_all("h1.kt-page-title__title.kt-page-title__title--responsive-sized")
        b = await page.query_selector_all("a.kt-unexpandable-row__action.kt-text-truncate") 
        info = await page.query_selector_all("p.kt-unexpandable-row__value")
        k = await page.query_selector_all("td.kt-group-row-item.kt-group-row-item__value.kt-group-row-item--info-row")
        s = await page.query_selector_all("div.kt-score-row__score")
        await page.wait_for_timeout(1000)
        for i in t  : 
            text = await i.inner_text()
            title.append(text) 
        for i in b : 
            text =await i.inner_text()
            brand.append(text)
        done = 0
        for i in info : 
            text = await i.inner_text()
            if done == 0 : 
                if re.search(r"ماه" , text):
                    bemaeh.append(text) 
            if re.search(r"دنده‌ای" , text): 
                gerbox.append(text)
            if re.search(r"اتوماتیک" , text): 
                gerbox.append(text)
            if re.search(r"اتومات" , text): 
                gerbox.append(text)
            if re.search(r"بنزین" , text): 
                fuel.append(text)
            if re.search(r"دوگانه‌سوز" , text): 
                fuel.append(text) 
            if re.search(r"تومان" , text): 
                price.append(text)
            done += 1
        done = 0     
        for i in k :
            text = await i.inner_text()
            if done == 0 :
                if re.search(r"[\d۰-۹٠-٩][\d۰-۹٠-٩,٬،\.]*", text):
                    km.append(text)
            if color_re.search(text):
                colors.append(text)
            done += 1
        done = 0    
        for i in s : 
            text = await i.inner_text()
            if done == 0 :
                motor_status.append(text)
            if done == 1 : 
                Chassis_status.append(text)
            if done == 2 : 
                body_status.append(text)
            if done == 3 : 
                gerbox_status.append(text)
            done += 1    
        await page.wait_for_timeout(300) 
        print(title)
        print(brand)
        print(gerbox)
        print(fuel)
        print(price)
        print(bemaeh)
        print(km)
        print(colors)
        print(motor_status)
        print(Chassis_status)
        print(body_status)
        print(gerbox_status)

In [ ]:
len(title)

In [ ]:
len(brand)

In [ ]:
len(gerbox)

In [ ]:
len(fuel)

In [ ]:
len(price)

In [ ]:
len(bemaeh)

In [ ]:
len(km)

In [ ]:
len(colors)

In [ ]:
len(motor_status)

In [ ]:
len(Chassis_status)

In [ ]:
len(body_status)

In [ ]:
len(gerbox_status)

In [ ]:
title = title[:1400]
brand = brand[:1400]
gerbox = gerbox[:1400]
fuel = fuel[:1400]
price = price[:1400]
bemaeh = bemaeh[:1400]
km = km[:1400]
colors = colors[:1400]
motor_status = motor_status[:1400]
Chassis_status = Chassis_status[:1400]
body_status = body_status[:1400]
gerbox_status = gerbox_status[:1400]

In [ ]:
df = pd.DataFrame({
    "title":title , 
    "brand":brand , 
    "gerbox":gerbox ,
    "fuel":fuel , 
    "price":price , 
    "bemaeh":bemaeh ,
    "km":km ,
    "colors":colors , 
    "motor_status":motor_status , 
    "chassis_status":Chassis_status , 
    "body_status":body_status , 
    "gerbox_status":gerbox_status
})

In [ ]:
df.tail(20)

In [ ]:
trans = str.maketrans("۰۱۲۳۴۵۶۷۸۹","0123456789")

In [ ]:
df['price'] = (
    df['price']
    .str.translate(trans)
    .str.replace(',' , '' ,regex=False)
    .str.replace("تومان", "", regex=False)
    .str.strip()
    .astype(int)
)

In [ ]:
df["km"] = (
    df["km"]
    .str.translate(trans)
    .str.strip()
    .astype(int)
)

In [ ]:
df["bemaeh"] = (
    df["bemaeh"]
    .str.translate(trans)
    .str.replace("ماه" , "" , regex=False)
    .str.strip()
    .astype(int)
)

In [ ]:
df.head()

In [ ]:
len(df["brand"].unique())

In [ ]:
df.to_csv("./divar_car_dataset.csv")